# ARC-AGI-3 BFS + Local Qwen Assist Submission Notebook

This notebook ignores `rgb-agent` and builds a self-contained ARC-AGI-3 submission around a custom agent.

## Strategy

- Use a local `BFS` planner first when game source files are available.
- Reuse previous level solutions via direct replay and coordinate-transfer for click actions.
- When search is unavailable or stalls, ask a locally loaded `Qwen2.5-72B-Instruct-AWQ` model for a short list of candidate actions.
- Fall back to a deterministic heuristic explorer if the local model is unavailable or unhelpful.
- Inject the agent into the official `ARC-AGI-3-Agents` harness during Kaggle competition reruns.

## LLM usage

The local Qwen model is only a helper. It does not control every move.

- Good use: narrowing click candidates, proposing action order, breaking out of repeated states.
- Safe fallback: if the model is unavailable, returns malformed JSON, or gives unusable actions, the agent immediately falls back to heuristics.

## Important

This notebook does not use `vLLM` and does not call the OpenAI API. The assist model is loaded directly with `transformers` from a local AWQ model directory.


In [ ]:
!pip install -q --no-index --find-links     /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels     arc-agi arc-agi-3


In [ ]:
!pip check


In [ ]:
%%writefile /kaggle/working/my_agent.py
import copy
import glob
import hashlib
import importlib.util
import json
import logging
import os
import random
import re
import time
from collections import Counter, deque
from pathlib import Path
from threading import Lock
from typing import Any, Optional

import numpy as np

from ..agent import Agent
from arcengine import ActionInput, FrameData, GameAction, GameState

logger = logging.getLogger(__name__)

SIMPLE_ACTION_IDS = [1, 2, 3, 4, 5, 7]


def _safe_int(value: Any, default: int = 0) -> int:
    try:
        return int(value)
    except Exception:
        return default


def _normalize_available_actions(actions: Any) -> list[int]:
    normalized: list[int] = []
    for action in actions or []:
        if hasattr(action, 'value'):
            try:
                normalized.append(int(action.value))
                continue
            except Exception:
                pass
        normalized.append(_safe_int(action, -1))
    return [a for a in normalized if a >= 0]


def _latest_grid(frame: FrameData) -> np.ndarray:
    payload = getattr(frame, 'frame', None)
    if payload is None:
        return np.zeros((64, 64), dtype=np.uint8)
    arr = np.asarray(payload, dtype=np.uint8)
    if arr.ndim == 3:
        return arr[-1]
    if arr.ndim == 2:
        return arr
    return np.zeros((64, 64), dtype=np.uint8)


def _level_idx(frame: FrameData) -> int:
    for field in ('levels_completed', 'score'):
        if hasattr(frame, field):
            return _safe_int(getattr(frame, field), 0)
    return 0


def _frame_hash(raw: np.ndarray) -> str:
    return hashlib.md5(raw.tobytes()).hexdigest()[:16]


def _background_color(raw: np.ndarray) -> int:
    if raw.size == 0:
        return 0
    return int(np.bincount(raw.reshape(-1), minlength=16).argmax())


def _build_simple_action(action_id: int, reason: str) -> GameAction:
    action = GameAction.from_id(int(action_id))
    action.reasoning = reason
    return action


def _build_click_action(x: int, y: int, reason: str) -> GameAction:
    action = GameAction.ACTION6
    action.set_data({'x': int(x), 'y': int(y)})
    action.reasoning = {'policy': reason, 'x': int(x), 'y': int(y)}
    return action


def _extract_json_payload(text: str) -> Optional[dict[str, Any]]:
    clean = re.sub(r"```(?:json)?\s*", "", text).replace("```", "").strip()
    decoder = json.JSONDecoder()
    for idx, char in enumerate(clean):
        if char != '{':
            continue
        try:
            parsed, _ = decoder.raw_decode(clean, idx)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            return parsed
    return None


class LocalQwenAssistant:
    def __init__(self) -> None:
        self.enabled = os.environ.get('LLM_ASSIST_ENABLED', '0') == '1'
        self.model_path = os.environ.get('LOCAL_QWEN_MODEL_PATH', '')
        self.max_new_tokens = int(os.environ.get('LLM_MAX_NEW_TOKENS', '220'))
        self.temperature = float(os.environ.get('LLM_TEMPERATURE', '0.0'))
        self.dtype_name = os.environ.get('LLM_TORCH_DTYPE', 'bfloat16')
        self.device_map = os.environ.get('LLM_DEVICE_MAP', 'auto')
        self.trust_remote_code = os.environ.get('LLM_TRUST_REMOTE_CODE', '1') == '1'

        self._torch = None
        self._tokenizer = None
        self._model = None
        self._model_lock = Lock()
        self._inference_lock = Lock()

    def available(self) -> bool:
        return self.enabled and bool(self.model_path)

    def _ensure_model(self) -> None:
        if self._model is not None and self._tokenizer is not None and self._torch is not None:
            return
        with self._model_lock:
            if self._model is not None and self._tokenizer is not None and self._torch is not None:
                return

            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer

            self._torch = torch
            self._tokenizer = AutoTokenizer.from_pretrained(
                self.model_path,
                trust_remote_code=self.trust_remote_code,
            )

            torch_dtype = torch.float16 if self.dtype_name == 'float16' else torch.bfloat16
            self._model = AutoModelForCausalLM.from_pretrained(
                self.model_path,
                trust_remote_code=self.trust_remote_code,
                device_map=self.device_map,
                torch_dtype=torch_dtype,
            )
            self._model.eval()

        logger.info('Loaded local Qwen assistant from %s', self.model_path)

    def _input_device(self):
        if self._model is None:
            raise RuntimeError('model not loaded')
        try:
            return self._model.get_input_embeddings().weight.device
        except Exception:
            pass
        try:
            return next(self._model.parameters()).device
        except Exception:
            return 'cuda:0'

    def _summarize_grid(self, raw: np.ndarray) -> dict[str, Any]:
        bg = _background_color(raw)
        objects = []
        h, w = raw.shape
        for color in range(16):
            if color == bg:
                continue
            ys, xs = np.where(raw == color)
            count = len(xs)
            if count < 2:
                continue
            objects.append({
                'color': int(color),
                'count': int(count),
                'cx': round(float(np.mean(xs)), 1),
                'cy': round(float(np.mean(ys)), 1),
                'bbox': [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
            })
        objects.sort(key=lambda item: (-item['count'], item['color']))
        return {
            'shape': [int(h), int(w)],
            'background': int(bg),
            'objects': objects[:12],
        }

    def suggest_actions(
        self,
        *,
        game_id: str,
        level_idx: int,
        raw: np.ndarray,
        available_ids: list[int],
        recent_actions: list[dict[str, Any]],
        changed_points: list[list[int]],
        repeated_state_count: int,
    ) -> list[dict[str, Any]]:
        if not self.available():
            return []

        try:
            self._ensure_model()
        except Exception as exc:
            logger.info('Local Qwen assistant unavailable: %s', exc)
            return []

        summary = self._summarize_grid(raw)
        prompt = {
            'game_id': game_id,
            'level_idx': level_idx,
            'available_actions': available_ids,
            'repeated_state_count': repeated_state_count,
            'recent_actions': recent_actions[-8:],
            'changed_points': changed_points[:16],
            'grid_summary': summary,
            'task': (
                'Propose up to 6 candidate next actions for an ARC-AGI-3 agent. '
                'Prefer actions that reveal mechanics or make progress. '
                'If ACTION6 is available, propose concrete click coordinates on meaningful objects or changed regions. '
                'Return only JSON with schema '
                '{"actions": [{"id": int, "x": int|null, "y": int|null, "reason": str}]}'
            ),
        }

        messages = [
            {
                'role': 'system',
                'content': (
                    'You are assisting a search agent for ARC-AGI-3. '
                    'Return strict JSON only. No markdown. No prose outside JSON.'
                ),
            },
            {'role': 'user', 'content': json.dumps(prompt, ensure_ascii=False)},
        ]

        try:
            rendered = self._tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            rendered = (
                'System: Return strict JSON only. No markdown. No prose outside JSON.\n\n'
                f'User: {json.dumps(prompt, ensure_ascii=False)}\n\nAssistant:'
            )

        with self._inference_lock:
            try:
                inputs = self._tokenizer(rendered, return_tensors='pt')
                inputs = {k: v.to(self._input_device()) for k, v in inputs.items()}
                with self._torch.no_grad():
                    outputs = self._model.generate(
                        **inputs,
                        max_new_tokens=self.max_new_tokens,
                        do_sample=self.temperature > 0,
                        temperature=max(self.temperature, 1e-5),
                        pad_token_id=self._tokenizer.eos_token_id,
                    )
                generated = outputs[0][inputs['input_ids'].shape[1]:]
                text = self._tokenizer.decode(generated, skip_special_tokens=True).strip()
            except Exception as exc:
                logger.info('Local Qwen generation failed for %s level %d: %s', game_id, level_idx, exc)
                return []

        parsed = _extract_json_payload(text)
        if not parsed:
            return []
        actions = parsed.get('actions', [])
        if not isinstance(actions, list):
            return []

        normalized = []
        for item in actions:
            if not isinstance(item, dict):
                continue
            action_id = _safe_int(item.get('id'), -1)
            if action_id not in available_ids:
                continue
            entry = {
                'id': action_id,
                'reason': str(item.get('reason', 'qwen-suggested'))[:200],
            }
            if action_id == 6:
                if item.get('x') is None or item.get('y') is None:
                    continue
                entry['x'] = int(max(0, min(raw.shape[1] - 1, _safe_int(item.get('x')))))
                entry['y'] = int(max(0, min(raw.shape[0] - 1, _safe_int(item.get('y')))))
            normalized.append(entry)
        return normalized[:6]


class BFSSolver:
    def __init__(self, game_path: str, class_name: str, *, scan_timeout: float, bfs_timeout: float, max_depth: int, max_states: int):
        self.game_path = game_path
        self.class_name = class_name
        self.scan_timeout = scan_timeout
        self.bfs_timeout = bfs_timeout
        self.max_depth = max_depth
        self.max_states = max_states
        self.game_cls = None
        self.solutions: dict[int, list[tuple[int, Optional[dict[str, int]]]]] = {}

    def load(self) -> bool:
        try:
            spec = importlib.util.spec_from_file_location('arc_game_mod', self.game_path)
            if spec is None or spec.loader is None:
                return False
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            self.game_cls = getattr(module, self.class_name)
            return True
        except Exception as exc:
            logger.warning('BFS load failed for %s: %s', self.game_path, exc)
            return False

    def _make_game(self, level_idx: int):
        game = self.game_cls()
        if hasattr(game, 'set_level'):
            game.set_level(level_idx)
        game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        return game

    def _run_action(self, game, action_id: int, data: Optional[dict[str, int]] = None):
        game_action = GameAction.RESET if int(action_id) == 0 else GameAction.from_id(action_id)
        ai = ActionInput(id=game_action, data=data) if data else ActionInput(id=game_action)
        return game.perform_action(ai, raw=True)

    def _scan_actions(self, game, raw0: np.ndarray, bg: int) -> list[tuple[int, Optional[dict[str, int]]]]:
        available = [a for a in getattr(game, '_available_actions', []) if a != 0]
        found: list[tuple[int, Optional[dict[str, int]]]] = []

        for action_id in [a for a in available if a in SIMPLE_ACTION_IDS]:
            g = copy.deepcopy(game)
            try:
                result = self._run_action(g, action_id)
            except Exception:
                continue
            frames = getattr(result, 'frame', None)
            if frames:
                raw = np.asarray(frames[-1], dtype=np.uint8)
                if raw.shape == raw0.shape and np.any(raw != raw0):
                    found.append((int(action_id), None))

        if 6 in available:
            seen_effects: set[str] = set()
            started = time.time()
            for y in range(0, raw0.shape[0], 2):
                if time.time() - started > self.scan_timeout:
                    break
                for x in range(0, raw0.shape[1], 2):
                    if raw0[y, x] == bg:
                        continue
                    g = copy.deepcopy(game)
                    payload = {'x': int(x), 'y': int(y)}
                    try:
                        result = self._run_action(g, 6, payload)
                    except Exception:
                        continue
                    frames = getattr(result, 'frame', None)
                    if not frames:
                        continue
                    raw = np.asarray(frames[-1], dtype=np.uint8)
                    if raw.shape != raw0.shape or not np.any(raw != raw0):
                        continue
                    effect = _frame_hash(raw)
                    if effect in seen_effects:
                        continue
                    seen_effects.add(effect)
                    found.append((6, payload))
        return found

    def _completion_value(self, result, game, fallback_level: int) -> int:
        if hasattr(result, 'levels_completed'):
            return _safe_int(getattr(result, 'levels_completed'), fallback_level)
        if hasattr(result, 'score'):
            return _safe_int(getattr(result, 'score'), fallback_level)
        if hasattr(game, '_current_level_index'):
            return _safe_int(getattr(game, '_current_level_index'), fallback_level)
        return fallback_level

    def _probe_hidden_fields(self, game, actions: list[tuple[int, Optional[dict[str, int]]]]) -> list[str]:
        if not actions:
            return []
        initial: dict[str, Any] = {}
        for key, value in game.__dict__.items():
            if isinstance(value, (int, float, bool)) and not key.startswith('__'):
                initial[key] = value
        changed: set[str] = set()
        for action_id, data in actions[:10]:
            g = copy.deepcopy(game)
            try:
                self._run_action(g, action_id, data)
            except Exception:
                continue
            for key, value in g.__dict__.items():
                if not isinstance(value, (int, float, bool)) or key.startswith('__'):
                    continue
                if key in initial and value != initial[key] and key not in {'_action_count', '_full_reset', '_action_complete'}:
                    changed.add(key)
        result = []
        for key in sorted(changed):
            if key.startswith('_') and key not in {'_current_level_index', '_score'}:
                continue
            result.append(key)
        return result

    def _try_transfer(self, level_idx: int, game, prev_solution: list[tuple[int, Optional[dict[str, int]]]], raw_current: np.ndarray):
        try:
            g = copy.deepcopy(game)
            for i, (action_id, data) in enumerate(prev_solution):
                try:
                    result = self._run_action(g, action_id, data)
                except Exception:
                    break
                if self._completion_value(result, g, level_idx) > level_idx:
                    solution = prev_solution[: i + 1]
                    self.solutions[level_idx] = solution
                    return solution

            prev_game = self._make_game(level_idx - 1)
            prev_reset = self._run_action(prev_game, 0)
            frames = getattr(prev_reset, 'frame', None)
            if not frames:
                return None
            raw_prev = np.asarray(frames[-1], dtype=np.uint8)
            bg_prev = _background_color(raw_prev)
            bg_curr = _background_color(raw_current)

            def objs(raw: np.ndarray, bg: int):
                found = []
                for color in range(16):
                    if color == bg:
                        continue
                    ys, xs = np.where(raw == color)
                    if len(xs) < 2:
                        continue
                    found.append({'color': color, 'cx': float(np.mean(xs)), 'cy': float(np.mean(ys)), 'n': int(len(xs))})
                return found

            source_objs = objs(raw_prev, bg_prev)
            target_objs = objs(raw_current, bg_curr)
            if not source_objs or not target_objs:
                return None

            matches = []
            for source in source_objs:
                best = None
                best_dist = float('inf')
                for target in target_objs:
                    if target['color'] != source['color']:
                        continue
                    if abs(target['n'] - source['n']) > max(target['n'], source['n']) * 0.5:
                        continue
                    dist = abs(target['cx'] - source['cx']) + abs(target['cy'] - source['cy'])
                    if dist < best_dist:
                        best = target
                        best_dist = dist
                if best is not None:
                    matches.append((source, best))
            if not matches:
                return None

            dx = float(np.mean([b['cx'] - a['cx'] for a, b in matches]))
            dy = float(np.mean([b['cy'] - a['cy'] for a, b in matches]))
            transferred = []
            for action_id, data in prev_solution:
                if action_id == 6 and data is not None:
                    shifted = {
                        'x': max(0, min(raw_current.shape[1] - 1, int(round(data['x'] + dx)))),
                        'y': max(0, min(raw_current.shape[0] - 1, int(round(data['y'] + dy)))),
                    }
                    transferred.append((action_id, shifted))
                else:
                    transferred.append((action_id, data))

            g = copy.deepcopy(game)
            for i, (action_id, data) in enumerate(transferred):
                try:
                    result = self._run_action(g, action_id, data)
                except Exception:
                    break
                if self._completion_value(result, g, level_idx) > level_idx:
                    solution = transferred[: i + 1]
                    self.solutions[level_idx] = solution
                    return solution
        except Exception as exc:
            logger.warning('BFS transfer failed: %s', exc)
        return None

    def solve_level(self, level_idx: int, prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None):
        if self.game_cls is None:
            return None

        game = self._make_game(level_idx)
        reset_result = self._run_action(game, 0)
        frames = getattr(reset_result, 'frame', None)
        if not frames:
            return None
        raw0 = np.asarray(frames[-1], dtype=np.uint8)
        bg = _background_color(raw0)

        if prev_solution and level_idx > 0:
            transferred = self._try_transfer(level_idx, game, prev_solution, raw0)
            if transferred:
                return transferred

        actions = self._scan_actions(game, raw0, bg)
        if not actions:
            return None

        visited: set[tuple[str, tuple[tuple[str, Any], ...]]] = set()
        queue = deque()

        def state_key(g, raw: np.ndarray, hidden_fields: Optional[list[str]] = None):
            hidden_fields = hidden_fields or []
            hidden = []
            for field in hidden_fields:
                hidden.append((field, getattr(g, field, None)))
            return (_frame_hash(raw), tuple(hidden))

        key0 = state_key(game, raw0)
        visited.add(key0)
        queue.append((game, [], 0))
        started = time.time()
        explored = 0

        while queue and explored < self.max_states and (time.time() - started) < self.bfs_timeout:
            g, history, depth = queue.popleft()
            for action_id, data in actions:
                try:
                    g2 = copy.deepcopy(g)
                    result = self._run_action(g2, action_id, data)
                except Exception:
                    continue
                explored += 1
                frames = getattr(result, 'frame', None)
                if not frames:
                    continue
                raw = np.asarray(frames[-1], dtype=np.uint8)
                key = state_key(g2, raw)
                if key in visited:
                    continue
                visited.add(key)
                new_history = history + [(action_id, data)]
                if self._completion_value(result, g2, level_idx) > level_idx:
                    self.solutions[level_idx] = new_history
                    return new_history
                if depth + 1 < self.max_depth:
                    queue.append((g2, new_history, depth + 1))

        hidden_fields = self._probe_hidden_fields(game, actions)
        if not hidden_fields:
            return None

        visited = set()
        queue = deque()
        key0 = state_key(game, raw0, hidden_fields)
        visited.add(key0)
        queue.append((game, [], 0))
        retry_started = time.time()

        while queue and explored < self.max_states and (time.time() - retry_started) < max(5.0, self.bfs_timeout * 0.5):
            g, history, depth = queue.popleft()
            for action_id, data in actions:
                try:
                    g2 = copy.deepcopy(g)
                    result = self._run_action(g2, action_id, data)
                except Exception:
                    continue
                explored += 1
                frames = getattr(result, 'frame', None)
                if not frames:
                    continue
                raw = np.asarray(frames[-1], dtype=np.uint8)
                key = state_key(g2, raw, hidden_fields)
                if key in visited:
                    continue
                visited.add(key)
                new_history = history + [(action_id, data)]
                if self._completion_value(result, g2, level_idx) > level_idx:
                    self.solutions[level_idx] = new_history
                    return new_history
                if depth + 1 < self.max_depth:
                    queue.append((g2, new_history, depth + 1))
        return None


def find_game_source_and_class(game_id: str) -> tuple[Optional[str], Optional[str]]:
    gid = game_id.split('-')[0]
    class_name = gid[0].upper() + gid[1:] if gid else None
    source_path = None

    patterns = [
        f'/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/{gid}/*/{gid}.py',
        f'/kaggle/input/**/{gid}/*/{gid}.py',
        f'/kaggle/working/**/{gid}.py',
        f'/tmp/*/{gid}/*/{gid}.py',
        f'**/game_sources/**/{gid}.py',
    ]
    for pattern in patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            source_path = matches[0]
            break

    if source_path is None:
        return None, class_name

    try:
        content = Path(source_path).read_text(encoding='utf-8', errors='ignore')[:4000]
        match = re.search(r'class\s+(\w+)\s*\(\s*ARCBaseGame', content)
        if match:
            class_name = match.group(1)
    except Exception:
        pass

    return source_path, class_name


class MyAgent(Agent):
    MAX_ACTIONS = int(os.environ.get('AGENT_MAX_ACTIONS', '140'))

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._rng = random.Random(int(os.environ.get('AGENT_RANDOM_SEED', '0')))
        self._scan_timeout = float(os.environ.get('BFS_SCAN_TIMEOUT', '1.5'))
        self._bfs_timeout = float(os.environ.get('BFS_TIMEOUT', '12.0'))
        self._bfs_depth = int(os.environ.get('BFS_MAX_DEPTH', '24'))
        self._bfs_states = int(os.environ.get('BFS_MAX_STATES', '60000'))

        self._game_id: Optional[str] = None
        self._current_level = -1
        self._solver: Optional[BFSSolver] = None
        self._llm = LocalQwenAssistant()
        self._solutions: dict[int, list[tuple[int, Optional[dict[str, int]]]]] = {}
        self._plan: list[tuple[int, Optional[dict[str, int]]]] = []
        self._plan_index = 0
        self._last_raw: Optional[np.ndarray] = None
        self._last_state_hash: Optional[str] = None
        self._state_visit_counts: Counter[str] = Counter()
        self._state_action_attempts: set[tuple[str, str]] = set()
        self._click_recency: deque[tuple[int, int]] = deque(maxlen=96)
        self._recent_actions: deque[dict[str, Any]] = deque(maxlen=24)
        self._llm_cache: dict[str, list[dict[str, Any]]] = {}
        self._llm_failed_states: set[str] = set()
        self._level_started_at = time.time()

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _reset_level_state(self, level_idx: int) -> None:
        self._current_level = level_idx
        self._plan = []
        self._plan_index = 0
        self._last_raw = None
        self._last_state_hash = None
        self._state_visit_counts.clear()
        self._state_action_attempts.clear()
        self._click_recency.clear()
        self._recent_actions.clear()
        self._llm_cache.clear()
        self._llm_failed_states.clear()
        self._level_started_at = time.time()

    def _ensure_solver(self, game_id: str) -> None:
        if self._solver is not None and self._game_id == game_id:
            return
        source_path, class_name = find_game_source_and_class(game_id)
        self._game_id = game_id
        if not source_path or not class_name:
            self._solver = None
            logger.info('No local game source found for %s; using Qwen/heuristic fallback.', game_id)
            return
        solver = BFSSolver(
            source_path,
            class_name,
            scan_timeout=self._scan_timeout,
            bfs_timeout=self._bfs_timeout,
            max_depth=self._bfs_depth,
            max_states=self._bfs_states,
        )
        self._solver = solver if solver.load() else None

    def _maybe_prepare_bfs_plan(self, level_idx: int) -> None:
        if self._solver is None or level_idx in self._solutions:
            return
        prev_solution = self._solutions.get(level_idx - 1)
        started = time.time()
        solution = self._solver.solve_level(level_idx, prev_solution=prev_solution)
        elapsed = time.time() - started
        if solution:
            self._solutions[level_idx] = solution
            self._plan = solution
            self._plan_index = 0
            logger.info('Prepared BFS plan for level %d in %.2fs (%d actions)', level_idx, elapsed, len(solution))
        else:
            logger.info('BFS could not solve level %d in %.2fs; using Qwen/heuristics.', level_idx, elapsed)

    def _non_background_points(self, raw: np.ndarray, bg: int) -> list[tuple[int, int]]:
        candidates: list[tuple[int, int]] = []
        h, w = raw.shape

        if self._last_raw is not None and self._last_raw.shape == raw.shape:
            diff = np.argwhere(raw != self._last_raw)
            if len(diff):
                center_y = int(np.median(diff[:, 0]))
                center_x = int(np.median(diff[:, 1]))
                candidates.append((center_x, center_y))
                for y, x in diff[:24]:
                    candidates.append((int(x), int(y)))

        for color in range(16):
            if color == bg:
                continue
            ys, xs = np.where(raw == color)
            count = len(xs)
            if count < 2:
                continue
            candidates.append((int(np.mean(xs)), int(np.mean(ys))))
            candidates.append((int(xs.min()), int(ys.min())))
            candidates.append((int(xs.max()), int(ys.max())))
            candidates.append((int(xs[len(xs) // 2]), int(ys[len(ys) // 2])))

        stride = 4
        for y in range(0, h, stride):
            for x in range(0, w, stride):
                if raw[y, x] != bg:
                    candidates.append((x, y))

        unique = []
        seen = set()
        for x, y in candidates:
            x = max(0, min(w - 1, int(x)))
            y = max(0, min(h - 1, int(y)))
            if (x, y) in seen:
                continue
            seen.add((x, y))
            unique.append((x, y))
        return unique[:96]

    def _changed_points(self, raw: np.ndarray) -> list[list[int]]:
        if self._last_raw is None or self._last_raw.shape != raw.shape:
            return []
        diff = np.argwhere(raw != self._last_raw)
        return [[int(x), int(y)] for y, x in diff[:24]]

    def _pick_click(self, state_hash: str, raw: np.ndarray) -> Optional[GameAction]:
        bg = _background_color(raw)
        candidates = self._non_background_points(raw, bg)
        if not candidates:
            return None

        def score(point: tuple[int, int]) -> tuple[int, int, int]:
            x, y = point
            used_in_state = 1 if (state_hash, f'6:{x}:{y}') in self._state_action_attempts else 0
            recent_penalty = 1 if point in self._click_recency else 0
            return (used_in_state, recent_penalty, abs(y - raw.shape[0] // 2) + abs(x - raw.shape[1] // 2))

        x, y = min(candidates, key=score)
        self._state_action_attempts.add((state_hash, f'6:{x}:{y}'))
        self._click_recency.append((x, y))
        return _build_click_action(x, y, 'heuristic-click')

    def _pick_simple(self, state_hash: str, available_ids: list[int]) -> Optional[GameAction]:
        preferred_order = {5: 0, 1: 1, 4: 2, 3: 3, 2: 4, 7: 5}
        simple_ids = [a for a in available_ids if a in SIMPLE_ACTION_IDS]
        if not simple_ids:
            return None

        def score(action_id: int) -> tuple[int, int]:
            used = 1 if (state_hash, str(action_id)) in self._state_action_attempts else 0
            return (used, preferred_order.get(action_id, 99))

        action_id = min(simple_ids, key=score)
        self._state_action_attempts.add((state_hash, str(action_id)))
        return _build_simple_action(action_id, 'heuristic-simple')

    def _llm_action(self, latest_frame: FrameData, raw: np.ndarray, state_hash: str) -> Optional[GameAction]:
        if not self._llm.available():
            return None
        if state_hash in self._llm_failed_states:
            return None

        visits = self._state_visit_counts[state_hash]
        elapsed = time.time() - self._level_started_at
        if visits < 2 and elapsed < 2.0:
            return None

        available_ids = _normalize_available_actions(getattr(latest_frame, 'available_actions', []))
        suggestions = self._llm_cache.get(state_hash)
        if suggestions is None:
            suggestions = self._llm.suggest_actions(
                game_id=self._game_id or getattr(latest_frame, 'game_id', '') or '',
                level_idx=self._current_level,
                raw=raw,
                available_ids=available_ids,
                recent_actions=list(self._recent_actions),
                changed_points=self._changed_points(raw),
                repeated_state_count=visits,
            )
            if not suggestions:
                self._llm_failed_states.add(state_hash)
                return None
            self._llm_cache[state_hash] = suggestions

        while suggestions:
            item = suggestions.pop(0)
            action_id = _safe_int(item.get('id'), -1)
            if action_id not in available_ids:
                continue
            if action_id == 6:
                x = _safe_int(item.get('x'), raw.shape[1] // 2)
                y = _safe_int(item.get('y'), raw.shape[0] // 2)
                key = f'6:{x}:{y}'
                if (state_hash, key) in self._state_action_attempts:
                    continue
                self._state_action_attempts.add((state_hash, key))
                self._click_recency.append((x, y))
                return _build_click_action(x, y, f"qwen:{item.get('reason', 'suggested')}")
            key = str(action_id)
            if (state_hash, key) in self._state_action_attempts:
                continue
            self._state_action_attempts.add((state_hash, key))
            return _build_simple_action(action_id, f"qwen:{item.get('reason', 'suggested')}")
        return None

    def _fallback_action(self, latest_frame: FrameData, raw: np.ndarray) -> GameAction:
        available_ids = _normalize_available_actions(getattr(latest_frame, 'available_actions', []))
        state_hash = _frame_hash(raw)
        visits = self._state_visit_counts[state_hash]

        llm_action = self._llm_action(latest_frame, raw, state_hash)
        if llm_action is not None:
            return llm_action

        click_action = self._pick_click(state_hash, raw) if 6 in available_ids else None
        simple_action = self._pick_simple(state_hash, available_ids)

        if visits >= 2 and click_action is not None:
            return click_action
        if simple_action is not None:
            return simple_action
        if click_action is not None:
            return click_action

        for action_id in available_ids:
            if action_id == 0:
                continue
            if action_id == 6:
                return _build_click_action(raw.shape[1] // 2, raw.shape[0] // 2, 'center-fallback')
            return _build_simple_action(action_id, 'available-fallback')

        return _build_simple_action(1, 'hard-fallback')

    def _record_action(self, action: GameAction) -> None:
        payload = {'id': _safe_int(getattr(action, 'value', -1), -1), 'reasoning': getattr(action, 'reasoning', None)}
        data = getattr(action, 'data', None)
        if data:
            payload['data'] = dict(data)
        self._recent_actions.append(payload)

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            action = GameAction.RESET
            action.reasoning = 'reset-on-start-or-game-over'
            self._record_action(action)
            return action

        raw = _latest_grid(latest_frame)
        level_idx = _level_idx(latest_frame)
        game_id = getattr(latest_frame, 'game_id', '') or ''

        if level_idx != self._current_level:
            self._reset_level_state(level_idx)
            self._ensure_solver(game_id)
            self._maybe_prepare_bfs_plan(level_idx)

        state_hash = _frame_hash(raw)
        self._state_visit_counts[state_hash] += 1

        if self._plan_index < len(self._plan):
            action_id, data = self._plan[self._plan_index]
            self._plan_index += 1
            available_ids = _normalize_available_actions(getattr(latest_frame, 'available_actions', []))
            if action_id == 6:
                if 6 in available_ids and data is not None:
                    action = _build_click_action(data['x'], data['y'], 'bfs-plan')
                    self._record_action(action)
                    return action
            elif action_id in available_ids:
                action = _build_simple_action(action_id, 'bfs-plan')
                self._record_action(action)
                return action
            self._plan = []
            self._plan_index = 0

        action = self._fallback_action(latest_frame, raw)
        self._last_raw = raw.copy()
        self._last_state_hash = state_hash
        self._record_action(action)
        return action


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if os.getenv("KAGGLE_IS_COMPETITION_RERUN") == "1":
    BASE = Path("/kaggle/working/ARC-AGI-3-Agents")
    SRC = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    AGENT_SRC = Path("/kaggle/working/my_agent.py")
    AGENT_DST = BASE / "agents/templates/my_agent.py"

    subprocess.run([
        "curl",
        "--fail",
        "--retry", "60",
        "--retry-all-errors",
        "--retry-delay", "5",
        "--retry-max-time", "300",
        "http://gateway:8001/api/games",
    ], check=False)

    if BASE.exists():
        shutil.rmtree(BASE)
    shutil.copytree(SRC, BASE)

    if not AGENT_SRC.exists():
        raise FileNotFoundError("my_agent.py not found in working directory")

    shutil.copy(AGENT_SRC, AGENT_DST)

    init_file = BASE / "agents/__init__.py"
    init_file.write_text(
        "from typing import Type\n"
        "from dotenv import load_dotenv\n"
        "from .agent import Agent, Playback\n"
        "from .swarm import Swarm\n"
        "from .templates.random_agent import Random\n"
        "from .templates.my_agent import MyAgent\n\n"
        "load_dotenv()\n\n"
        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
        '    "random": Random,\n'
        '    "myagent": MyAgent\n'
        "}\n"
    )

    env_file = BASE / ".env"
    env_file.write_text(
        "\n".join([
            "SCHEME=http",
            "HOST=gateway",
            "PORT=8001",
            "ARC_API_KEY=test-key-123",
            "ARC_BASE_URL=http://gateway:8001/",
            "OPERATION_MODE=online",
            "RECORDINGS_DIR=/kaggle/working/server_recording",
            "AGENT_MAX_ACTIONS=140",
            "BFS_SCAN_TIMEOUT=1.5",
            "BFS_TIMEOUT=12.0",
            "BFS_MAX_DEPTH=24",
            "BFS_MAX_STATES=60000",
            "LLM_ASSIST_ENABLED=0",
            "LOCAL_QWEN_MODEL_PATH=/kaggle/input/your-qwen25-72b-awq-model/Qwen2.5-72B-Instruct-AWQ",
            "LLM_MAX_NEW_TOKENS=220",
            "LLM_TEMPERATURE=0.0",
            "LLM_TORCH_DTYPE=bfloat16",
            "LLM_DEVICE_MAP=auto",
            "LLM_TRUST_REMOTE_CODE=1",
        ]) + "\n"
    )

    subprocess.run(
        ["python", "main.py", "--agent", "myagent"],
        cwd=str(BASE),
        env={**os.environ, "MPLBACKEND": "agg", "PYTHONUNBUFFERED": "1"},
        check=True,
    )
else:
    print("Competition rerun not detected. Agent file is ready at /kaggle/working/my_agent.py")
    print("To enable local Qwen assist later, set LLM_ASSIST_ENABLED=1 and point LOCAL_QWEN_MODEL_PATH to the local AWQ model directory.")


In [ ]:
import os

if os.getenv("KAGGLE_IS_COMPETITION_RERUN") != "1":
    import pandas as pd

    submission = pd.DataFrame([
        {
            "row_id": "debug_0",
            "game_id": "1",
            "end_of_game": True,
            "score": 0,
        }
    ])
    submission.to_parquet("/kaggle/working/submission.parquet", index=False)
